# 🤖 Txoko RAG Chatbot — Guía de implementación
### Bootcamp BBK The Bridge · Equipo 4
**Autores:** Naia, Andoni, Unai, Fátima  
**Fecha:** Junio 2026

---

## ¿Qué es este notebook?

Este notebook documenta paso a paso cómo hemos añadido un **chatbot de lenguaje natural** a la app Txoko, una aplicación de recomendaciones turísticas para Euskadi.

El chatbot permite consultas como:
- *"Recomiéndame un restaurante de pintxos cerca del Guggenheim"*
- *"¿Qué playas tranquilas hay en Gipuzkoa?"*
- *"Busco alojamiento rural en el campo"*

---

## Índice

1. [¿Qué es RAG y por qué lo usamos?](#1-rag)
2. [Arquitectura del sistema](#2-arquitectura)
3. [Instalación de dependencias](#3-dependencias)
4. [Paso 1 — Generar embeddings y construir el índice FAISS](#4-embeddings)
5. [Paso 2 — Verificar que el índice funciona](#5-verificacion)
6. [Próximos pasos](#6-proximos)


## 1. ¿Qué es RAG y por qué lo usamos? <a name="1-rag"></a>

### El problema

Queremos un chatbot que responda preguntas sobre lugares turísticos de Euskadi usando **nuestra propia base de datos**.

Las dos opciones que podríamos considerar:

| Opción | Descripción | ¿Por qué no? |
|--------|-------------|--------------|
| **Entrenar un modelo desde cero** | Crear nuestro propio LLM con nuestros datos | Requiere millones de ejemplos, semanas de cómputo y cientos de miles de euros en GPUs |
| **Fine-tuning** | Ajustar un modelo existente con nuestros datos | Sigue requiriendo muchos datos etiquetados y es costoso. El modelo "memoriza" datos estáticos y se queda desactualizado |
| **RAG ✅** | Usar un LLM existente + buscar en nuestra BD en tiempo real | Económico, actualizable, no requiere entrenamiento |

### ¿Qué significa RAG?

**RAG = Retrieval-Augmented Generation** (Generación Aumentada por Recuperación)

La idea es sencilla: en vez de pedirle al LLM que "sepa" cosas de Euskadi, le proporcionamos la información relevante en cada consulta.

```
Usuario pregunta: "pintxos cerca del Guggenheim"
        ↓
[1] Nuestro código busca en la BD los lugares más relevantes
        ↓
[2] Nuestro código construye un prompt con esos lugares
        ↓
[3] El LLM genera una respuesta en lenguaje natural
        ↓
Usuario recibe: "Te recomiendo el Bar Txiriboga, a 200m del Guggenheim..."
```

**Clave conceptual:** El LLM no consulta nuestra base de datos directamente. Es nuestro código el que orquesta todo. El LLM solo recibe texto y genera texto.

### ¿Por qué no simplemente buscar en la BD con SQL?

Una búsqueda SQL normal (`WHERE nombre LIKE '%pintxos%'`) solo encuentra coincidencias exactas de texto.

RAG usa **búsqueda semántica**: entiende que "comida vasca tradicional", "pintxos" y "gastronomía del País Vasco" son conceptos relacionados, aunque no compartan palabras.


## 2. Arquitectura del sistema <a name="2-arquitectura"></a>

El sistema tiene dos fases bien diferenciadas:

### Fase offline (se ejecuta una vez)

```
aupa_master_v5.csv  ──┐
                       ├──► build_faiss_index.py ──► faiss_index.index
txoko_reviews_raw.json ┘                          └──► faiss_metadata.json
```

Convertimos todos nuestros lugares a **embeddings** (vectores numéricos) y los almacenamos en un índice FAISS. Esto es lo que hace este notebook.

### Fase online (se ejecuta en cada consulta)

```
Pregunta del usuario
        ↓
Convertir pregunta a embedding (mismo modelo)
        ↓
Buscar los k lugares más similares en FAISS
        ↓
Construir prompt con los lugares recuperados
        ↓
Llamar a la API de Gemini
        ↓
Devolver respuesta al usuario
```

### ¿Qué es un embedding?

Un embedding es una representación numérica de un texto como un vector de números.

```
"restaurante de pintxos en Bilbao" → [0.23, -0.45, 0.12, 0.87, ...]  (384 números)
"bar de pintxos bilbaíno"          → [0.21, -0.43, 0.14, 0.85, ...]  (muy similar)
"playa en Zarautz"                 → [0.67,  0.12, -0.34, 0.23, ...]  (muy diferente)
```

Textos con significado similar tienen vectores similares (cercanos en el espacio vectorial). Esto es lo que permite la búsqueda semántica.

### ¿Qué es FAISS?

FAISS (Facebook AI Similarity Search) es una librería que permite buscar eficientemente los vectores más similares a uno dado entre millones de vectores. En nuestro caso con ~4.600 lugares es más que suficiente con su configuración más simple.

El índice FAISS es simplemente **un fichero en disco** que se carga en memoria al arrancar el servidor. No es una base de datos con servidor propio.

### Componentes y tecnologías

| Componente | Tecnología | Por qué |
|------------|-----------|---------|
| Modelo de embeddings | `sentence-transformers` | Open source, multilingüe, corre en CPU |
| Índice vectorial | FAISS | Simple, sin servidor, fichero en disco |
| LLM | Gemini API (Google) | Tier gratuito generoso, buena calidad |
| Backend | FastAPI | Ligero, fácil de desplegar en Render |
| Despliegue | Render | Ya lo usamos para el resto del proyecto |


## 3. Instalación de dependencias <a name="3-dependencias"></a>

### Requisitos previos

- Python 3.9 o superior
- Los ficheros `aupa_master_v5.csv` y `txoko_reviews_raw.json` en la misma carpeta
- Una API key de Gemini (gratuita en [aistudio.google.com](https://aistudio.google.com))

### ⚠️ Importante: gestión segura de la API key

**Nunca escribas tu API key directamente en el código ni en un chat.**

Crea un fichero `.env` en la carpeta del proyecto:
```
GEMINI_API_KEY=tu_clave_aqui
```

Y añade `.env` a tu `.gitignore` para que nunca llegue al repositorio:
```
echo ".env" >> .gitignore
```


In [ ]:
# Instalar todas las dependencias necesarias
# Ejecutar esta celda solo una vez

# Si trabajas en local con entorno virtual:
# python -m venv venv
# source venv/bin/activate  (Mac/Linux)
# venv\Scripts\activate    (Windows PowerShell)

!pip install sentence-transformers faiss-cpu pandas numpy python-dotenv google-generativeai fastapi uvicorn
print("✓ Dependencias instaladas")


## 4. Paso 1 — Generar embeddings y construir el índice FAISS <a name="4-embeddings"></a>

### ¿Qué hace este paso?

1. Carga el CSV con los lugares y el JSON con las reseñas
2. Para cada lugar, construye un texto combinando sus campos + sus reseñas
3. Convierte cada texto en un embedding (vector de 384 números)
4. Almacena todos los embeddings en un índice FAISS
5. Guarda el índice y los metadatos en disco

### ¿Por qué combinamos reseñas con la descripción?

La descripción oficial de un lugar suele ser genérica:
> *"Restaurante de cocina vasca tradicional"*

Una reseña real añade vocabulario natural:
> *"Las croquetas son increíbles, perfecto para una cena tranquila en familia"*

El segundo texto responde mucho mejor a consultas como "cena tranquila" o "comida casera".

### ¿Por qué filtramos reseñas con rating < 3?

Para evitar que vocabulario negativo ("sucio", "lento", "caro") quede asociado semánticamente al lugar y aparezca en búsquedas que no corresponden.

### El modelo de embeddings

Usamos `paraphrase-multilingual-MiniLM-L12-v2`:
- **Multilingüe**: funciona en español, euskera e inglés
- **Ligero**: 384 dimensiones, corre en CPU sin problema  
- **Primera ejecución**: descarga ~120MB del modelo (se cachea localmente)


In [ ]:
import json
import os
import time

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# ─── Configuración ────────────────────────────────────────────────────────────

CSV_PATH      = "aupa_master_v5.csv"
REVIEWS_PATH  = "txoko_reviews_raw.json"
OUTPUT_DIR    = "./rag_assets"

MODEL_NAME    = "paraphrase-multilingual-MiniLM-L12-v2"
MAX_REVIEWS   = 5   # máximo de reseñas por lugar que se incorporan al texto

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✓ Configuración lista")


In [ ]:
# ─── Funciones de construcción de texto ──────────────────────────────────────

def load_reviews(reviews_path: str) -> dict:
    """
    Carga el JSON de reseñas y lo transforma en un dict:
        {id_lugar: [{"rating": 5, "text": "..."}, ...]}
    """
    with open(reviews_path, encoding="utf-8") as f:
        raw = json.load(f)

    reviews_by_id = {}
    for place_id, data in raw.items():
        if isinstance(data, dict) and "reviews" in data:
            reviews_by_id[place_id] = data["reviews"]
        elif isinstance(data, list):
            reviews_by_id[place_id] = data

    print(f"  Reseñas cargadas: {len(reviews_by_id):,} lugares con reseñas")
    return reviews_by_id


def build_place_text(row: pd.Series, reviews_by_id: dict) -> str:
    """
    Construye el texto que se convertirá en embedding para un lugar.
    
    Combina campos estructurados del CSV con reseñas reales de Google.
    Este texto es lo que el modelo 'lee' para entender de qué trata cada lugar.
    """
    parts = []

    parts.append(f"Nombre: {row.get('nombre', '')}")
    parts.append(f"Tipo: {row.get('subcategoria', '')} | {row.get('categoria', '')}")
    parts.append(f"Ubicación: {row.get('municipio', '')}, {row.get('territorio', '')}")

    desc = str(row.get("descripcion", "")).strip()
    if desc and desc != "nan":
        parts.append(f"Descripción: {desc}")

    # Incorporar reseñas (solo las positivas para evitar ruido semántico negativo)
    place_id = str(row.get("id", ""))
    if place_id in reviews_by_id:
        texts = [
            r["text"].strip()
            for r in reviews_by_id[place_id][:MAX_REVIEWS]
            if r.get("text", "").strip() and r.get("rating", 0) >= 3
        ]
        if texts:
            parts.append("Reseñas: " + " | ".join(texts))

    return "\n".join(parts)


def build_metadata(row: pd.Series) -> dict:
    """
    Extrae los metadatos que se guardan junto al índice.
    Son los datos que el backend devuelve al frontend con cada recomendación.
    """
    def safe(val):
        return None if pd.isna(val) else val

    return {
        "id":                 str(row.get("id", "")),
        "nombre":             str(row.get("nombre", "")),
        "categoria":          str(row.get("categoria", "")),
        "subcategoria":       str(row.get("subcategoria", "")),
        "municipio":          str(row.get("municipio", "")),
        "territorio":         str(row.get("territorio", "")),
        "lat":                safe(row.get("lat")),
        "lon":                safe(row.get("lon")),
        "descripcion":        str(row.get("descripcion", "")),
        "google_rating":      safe(row.get("google_rating")),
        "google_num_reviews": safe(row.get("google_num_reviews")),
        "web":                str(row.get("web", "")),
        "ficha_turismo":      str(row.get("ficha_turismo", "")),
    }

print("✓ Funciones definidas")


In [ ]:
# ─── 1. Cargar datos ──────────────────────────────────────────────────────────

print("[1/5] Cargando datos...")
df = pd.read_csv(CSV_PATH)
print(f"  CSV cargado: {len(df):,} lugares × {len(df.columns)} columnas")

# Filtrar lugares sin nombre (no son útiles para el chatbot)
df = df[df["nombre"].notna() & df["nombre"].str.strip().ne("")]
print(f"  Tras filtro nombre: {len(df):,} lugares")

reviews_by_id = load_reviews(REVIEWS_PATH)


In [ ]:
# ─── 2. Construir textos ──────────────────────────────────────────────────────

print("\n[2/5] Construyendo textos para embeddings...")
texts    = []
metadata = []

for _, row in df.iterrows():
    texts.append(build_place_text(row, reviews_by_id))
    metadata.append(build_metadata(row))

# Mostrar ejemplo del primer lugar para verificar que el texto tiene buena pinta
print("\n  Ejemplo — texto para embedding del primer lugar:")
print("  " + "\n  ".join(texts[0].split("\n")))

n_con_reviews = sum(1 for m in metadata if m["id"] in reviews_by_id)
print(f"\n  Lugares con reseñas incorporadas: {n_con_reviews:,} / {len(texts):,}")


In [ ]:
# ─── 3. Generar embeddings ────────────────────────────────────────────────────
#
# Este paso puede tardar entre 3 y 10 minutos en CPU.
# El modelo se descarga la primera vez (~120MB) y se cachea localmente.

print(f"\n[3/5] Cargando modelo '{MODEL_NAME}'...")
model = SentenceTransformer(MODEL_NAME)
print(f"  Dimensión de embeddings: {model.get_sentence_embedding_dimension()}")

print(f"\n  Generando embeddings para {len(texts):,} textos...")
t0 = time.time()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,  # necesario para similitud coseno con IndexFlatIP
)

print(f"\n  Embeddings generados en {time.time()-t0:.1f}s")
print(f"  Shape: {embeddings.shape}")  # esperado: (n_lugares, 384)


In [ ]:
# ─── 4. Construir índice FAISS ────────────────────────────────────────────────
#
# IndexFlatIP: búsqueda exacta por producto interior
# Con vectores normalizados equivale a similitud coseno.
# Para ~4.600 lugares es suficiente sin índices aproximados.
#
# Score de similitud coseno:
#   1.0  = idénticos
#   0.8+ = muy similar
#   0.6+ = relacionado
#   < 0.5 = poco relevante

print("\n[4/5] Construyendo índice FAISS...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype(np.float32))
print(f"  Vectores en el índice: {index.ntotal:,}")


In [ ]:
# ─── 5. Guardar outputs ───────────────────────────────────────────────────────

print("\n[5/5] Guardando outputs...")

index_path    = os.path.join(OUTPUT_DIR, "faiss_index.index")
metadata_path = os.path.join(OUTPUT_DIR, "faiss_metadata.json")

faiss.write_index(index, index_path)
print(f"  Índice FAISS : {index_path}  ({os.path.getsize(index_path)//1024} KB)")

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"  Metadatos    : {metadata_path}  ({os.path.getsize(metadata_path)//1024} KB)")

print(f"\n✓ Índice construido correctamente")
print(f"  Estos dos ficheros son todo lo que necesita el backend RAG:")
print(f"    - faiss_index.index      → el índice vectorial")
print(f"    - faiss_metadata.json    → metadatos de cada lugar (mismo orden que el índice)")


## 5. Paso 2 — Verificar que el índice funciona <a name="5-verificacion"></a>

Antes de construir el backend, comprobamos que las búsquedas semánticas devuelven resultados con sentido.

### ¿Qué buscamos al verificar?

- Que los scores sean razonables (> 0.7 para consultas claras)
- Que los resultados sean semánticamente coherentes con la pregunta
- Que haya diversidad de categorías cuando la pregunta es genérica

### Interpretación de scores

| Score | Interpretación |
|-------|---------------|
| > 0.80 | Muy alta similitud |
| 0.70 - 0.80 | Buena similitud |
| 0.60 - 0.70 | Similitud moderada |
| < 0.50 | Baja similitud — resultados poco fiables |

### Limitaciones conocidas

- **Nombres propios** ("Guggenheim"): FAISS puede no encontrarlos bien si la descripción en el CSV es pobre. Solución futura: búsqueda híbrida (semántica + texto exacto).
- **Categorías poco representadas** ("actividades para niños"): si los datos no tienen esa etiqueta, el modelo no tiene de dónde tirar. Scores bajos son una señal de alerta.
- **El LLM compensa parte de estos fallos**: recuperamos k=10 candidatos y dejamos que el LLM seleccione los más relevantes.


In [ ]:
# ─── Función de búsqueda para pruebas ────────────────────────────────────────

def buscar(pregunta: str, k: int = 5):
    """
    Convierte la pregunta a embedding, busca en FAISS y muestra los k resultados
    más similares con su score de similitud coseno.
    """
    print(f"\nPregunta: '{pregunta}'")
    print("-" * 55)

    # Mismo modelo y misma normalización que al construir el índice
    embedding = model.encode([pregunta], normalize_embeddings=True)
    scores, indices = index.search(embedding.astype(np.float32), k)

    for score, idx in zip(scores[0], indices[0]):
        lugar = metadata[idx]
        rating = f" ⭐{lugar['google_rating']}" if lugar.get("google_rating") else ""
        print(f"  [{score:.3f}]{rating} {lugar['nombre']} — {lugar['municipio']} ({lugar['categoria']})")

print("✓ Función buscar() lista")


In [ ]:
# ─── Pruebas de búsqueda ─────────────────────────────────────────────────────
# Modifica estas consultas para probar lo que necesites

buscar("restaurante de pintxos en Bilbao")
buscar("playa tranquila en Gipuzkoa")
buscar("museo de arte contemporáneo")
buscar("alojamiento rural en el campo")
buscar("actividades para niños")
buscar("Guggenheim Bilbao")
buscar("museo arte moderno Bilbao")


In [ ]:
# ─── Análisis de cobertura de reseñas ────────────────────────────────────────
# Cuántos lugares tienen reseñas incorporadas y cómo se distribuyen por categoría

df_meta = pd.DataFrame(metadata)
df_meta["tiene_reviews"] = df_meta["id"].isin(reviews_by_id.keys())

print("Cobertura de reseñas por categoría:")
print()
resumen = (df_meta.groupby("categoria")
           .agg(
               total=("id", "count"),
               con_reviews=("tiene_reviews", "sum"),
           ))
resumen["pct_reviews"] = (resumen["con_reviews"] / resumen["total"] * 100).round(1)
print(resumen.sort_values("total", ascending=False).to_string())


## 6. Próximos pasos <a name="6-proximos"></a>

Con el índice FAISS construido y verificado, los siguientes pasos son:

### Paso 3 — Backend FastAPI

Un servidor que expone un endpoint `/chat` y ejecuta el pipeline completo:

```
POST /chat
{"pregunta": "restaurante de pintxos cerca del Guggenheim"}

→ {"respuesta": "Te recomiendo el Bar Txiriboga, situado a 200m del Guggenheim..."}
```

### Paso 4 — Diseño del prompt

El system prompt define el comportamiento del chatbot. Es la pieza donde más se puede afinar la calidad de las respuestas:

```
Eres Txoko, un asistente de recomendaciones turísticas de Euskadi.
Responde SIEMPRE en el idioma en que te hablen.
Usa SOLO la información de los lugares que te proporciono.
Si no tienes información suficiente, dilo claramente.
No inventes horarios, precios ni distancias.
```

### Paso 5 — Despliegue en Render

Los ficheros `faiss_index.index` y `faiss_metadata.json` se suben junto al código del backend. Render carga el índice en memoria al arrancar.

**Consideración importante con el tier gratuito de Render:** el servicio se "duerme" tras inactividad. Al despertar tarda unos segundos en cargar el índice. Gestionar esto en el frontend con un mensaje de "conectando..." evita que parezca un error.

---

### requirements.txt para Render

```
fastapi
uvicorn
sentence-transformers
faiss-cpu
numpy
pandas
python-dotenv
google-generativeai - no es esta
```
